In [0]:
# ============================================================
# Notebook  : aggregate_flight_prices
# Purpose   : Read Silver table, calculate 7-day moving
#             average per route per airline per cabin class
#             and SCD2 MERGE into Gold table
# Author    : FlightPriceTracker
# Version   : v1.0
# ============================================================

import uuid
import traceback
from datetime import datetime, timezone, timedelta
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit, avg, min, max, stddev,
    count, current_timestamp, when, last
)
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    DoubleType, BooleanType, TimestampType
)
from pyspark.sql.window import Window
from delta.tables import DeltaTable

# ============================================================
# SECTION 1 — INIT
# ============================================================

spark = SparkSession.builder \
    .appName("FlightPriceTracker_Aggregate") \
    .getOrCreate()

# ============================================================
# SECTION 2 — CONSTANTS
# ============================================================

RUN_ID = spark.sql("SELECT run_id FROM workspace.flight_tracker.flight_prices_raw ORDER BY ingested_at DESC LIMIT 1").collect()[0][0]
JOB_NAME      = "FlightPriceTracker"
NOTEBOOK_NAME = "aggregate_flight_prices"
TASK_SEQUENCE = 3
TRIGGERED_BY  = "SCHEDULER"
DATABASE      = "workspace.flight_tracker"
CLEAN_TABLE   = f"{DATABASE}.flight_prices_clean"
AVG_TABLE     = f"{DATABASE}.flight_prices_avg"
AUDIT_TABLE   = f"{DATABASE}.pipeline_audit_log"
LOOKBACK_DAYS = 7

def now():
    return datetime.now(timezone.utc)

NOW            = now()
SEVEN_DAYS_AGO = NOW - timedelta(days=LOOKBACK_DAYS)

print(f"RUN_ID     : {RUN_ID}")
print(f"NOTEBOOK   : {NOTEBOOK_NAME}")
print(f"START TIME : {NOW}")

# ============================================================
# SECTION 3 — AUDIT SCHEMA
# ============================================================

audit_schema = StructType([
    StructField("run_id",               StringType(),    True),
    StructField("job_name",             StringType(),    True),
    StructField("notebook_name",        StringType(),    True),
    StructField("task_sequence",        IntegerType(),   True),
    StructField("cluster_id",           StringType(),    True),
    StructField("start_time",           TimestampType(), True),
    StructField("end_time",             TimestampType(), True),
    StructField("duration_seconds",     IntegerType(),   True),
    StructField("status",               StringType(),    True),
    StructField("records_read",         IntegerType(),   True),
    StructField("records_written",      IntegerType(),   True),
    StructField("records_skipped",      IntegerType(),   True),
    StructField("records_failed",       IntegerType(),   True),
    StructField("error_message",        StringType(),    True),
    StructField("error_stacktrace",     StringType(),    True),
    StructField("retry_attempt",        IntegerType(),   True),
    StructField("triggered_by",         StringType(),    True),
    StructField("databricks_job_run_id",StringType(),    True),
    StructField("created_at",           TimestampType(), True),
])

# ============================================================
# SECTION 4 — AUDIT HELPER
# ============================================================

def log_audit(
    run_id, job_name, notebook_name, task_sequence,
    start_time, end_time, status,
    records_read=0, records_written=0,
    records_skipped=0, records_failed=0,
    error_message=None, error_stacktrace=None,
    retry_attempt=0, triggered_by="SCHEDULER"
):
    duration = int((end_time - start_time).total_seconds())
    audit_data = [{
        "run_id"               : run_id,
        "job_name"             : job_name,
        "notebook_name"        : notebook_name,
        "task_sequence"        : task_sequence,
        "cluster_id"           : "serverless",
        "start_time"           : start_time,
        "end_time"             : end_time,
        "duration_seconds"     : duration,
        "status"               : status,
        "records_read"         : records_read,
        "records_written"      : records_written,
        "records_skipped"      : records_skipped,
        "records_failed"       : records_failed,
        "error_message"        : error_message if error_message else "",
        "error_stacktrace"     : error_stacktrace if error_stacktrace else "",
        "retry_attempt"        : retry_attempt,
        "triggered_by"         : triggered_by,
        "databricks_job_run_id": "",
        "created_at"           : now()
    }]
    audit_df = spark.createDataFrame(audit_data, schema=audit_schema)
    audit_df.write.format("delta").mode("append").saveAsTable(AUDIT_TABLE)
    print(f"[AUDIT] {status} logged for {notebook_name}")

# ============================================================
# SECTION 5 — COMPUTE AGGREGATIONS
# ============================================================

def compute_aggregations(df):
    window_spec = Window.partitionBy("route_id", "airline_code", "cabin_class") \
                        .orderBy(col("ingested_at").cast("long")) \
                        .rangeBetween(-LOOKBACK_DAYS * 86400, 0)

    latest_price_window = Window.partitionBy("route_id", "airline_code", "cabin_class") \
                                .orderBy(col("ingested_at").desc())

    agg_df = df \
        .withColumn("avg_7day_price",    avg("price_usd").over(window_spec)) \
        .withColumn("min_7day_price",    min("price_usd").over(window_spec)) \
        .withColumn("max_7day_price",    max("price_usd").over(window_spec)) \
        .withColumn("stddev_7day_price", stddev("price_usd").over(window_spec)) \
        .withColumn("data_points_count", count("price_usd").over(window_spec)) \
        .withColumn("latest_price",      last("price_usd").over(latest_price_window))

    agg_df = agg_df.withColumn(
        "price_drop_pct",
        when(
            col("avg_7day_price").isNotNull() & (col("avg_7day_price") > 0),
            ((col("avg_7day_price") - col("latest_price")) / col("avg_7day_price")) * 100
        ).otherwise(lit(0.0))
    )

    agg_df = agg_df.withColumn(
        "price_trend",
        when(col("price_drop_pct") >= 5,   lit("FALLING"))
        .when(col("price_drop_pct") <= -5, lit("RISING"))
        .otherwise(lit("STABLE"))
    )

    return agg_df.select(
        "route_id",
        "airline",
        "airline_code",
        "cabin_class",
        "avg_7day_price",
        "min_7day_price",
        "max_7day_price",
        "stddev_7day_price",
        "latest_price",
        "price_drop_pct",
        "price_trend",
        "data_points_count"
    ).dropDuplicates(["route_id", "airline_code", "cabin_class"])

# ============================================================
# SECTION 6 — SCD2 MERGE INTO GOLD
# ============================================================

def merge_into_gold(agg_df):
    now_ts = lit(NOW)

    new_df = agg_df \
        .withColumn("calculated_at", now_ts) \
        .withColumn("valid_from",    now_ts) \
        .withColumn("valid_to",      lit(None).cast("timestamp")) \
        .withColumn("is_current",    lit(True))

    gold_table = DeltaTable.forName(spark, AVG_TABLE)

    gold_table.alias("target").merge(
        new_df.alias("source"),
        """
        target.route_id     = source.route_id     AND
        target.airline_code = source.airline_code AND
        target.cabin_class  = source.cabin_class  AND
        target.is_current   = true
        """
    ).whenMatchedUpdate(
        condition = """
            target.latest_price   != source.latest_price OR
            target.avg_7day_price != source.avg_7day_price
        """,
        set = {
            "target.is_current" : lit(False),
            "target.valid_to"   : now_ts
        }
    ).whenNotMatchedInsertAll() \
     .execute()

    new_df.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(AVG_TABLE)

    print(f"[MERGE] SCD2 merge into {AVG_TABLE} complete")

# ============================================================
# SECTION 7 — MAIN EXECUTION
# ============================================================

start_time      = now()
records_read    = 0
records_written = 0
records_skipped = 0
records_failed  = 0
pipeline_status = "SUCCESS"
pipeline_error  = None
pipeline_trace  = None

try:
    print(f"\n[READ] Reading last {LOOKBACK_DAYS} days from {CLEAN_TABLE}...")
    clean_df = spark.sql(f"""
        SELECT * FROM {CLEAN_TABLE}
        WHERE is_valid     = true
        AND   is_duplicate = false
        AND   ingested_at >= '{SEVEN_DAYS_AGO}'
    """)

    records_read = clean_df.count()
    print(f"[READ] Records found: {records_read}")

    if records_read == 0:
        print("[READ] No records found for aggregation.")
        pipeline_status = "PARTIAL"

    else:
        agg_df          = compute_aggregations(clean_df)
        records_written = agg_df.count()

        merge_into_gold(agg_df)

        print(f"[SUMMARY] Read: {records_read} | Written: {records_written}")

except Exception as e:
    pipeline_status = "FAILED"
    pipeline_error  = str(e)
    pipeline_trace  = traceback.format_exc()
    print(f"[ERROR] {pipeline_error}")

finally:
    end_time = now()
    log_audit(
        run_id           = RUN_ID,
        job_name         = JOB_NAME,
        notebook_name    = NOTEBOOK_NAME,
        task_sequence    = TASK_SEQUENCE,
        start_time       = start_time,
        end_time         = end_time,
        status           = pipeline_status,
        records_read     = records_read,
        records_written  = records_written,
        records_skipped  = records_skipped,
        records_failed   = records_failed,
        error_message    = pipeline_error,
        error_stacktrace = pipeline_trace,
        triggered_by     = TRIGGERED_BY
    )
    print(f"\n[DONE] {NOTEBOOK_NAME} — Status: {pipeline_status}")
    print(f"[DONE] Read: {records_read} | Written: {records_written} | Skipped: {records_skipped} | Failed: {records_failed}")
    dbutils.notebook.exit(RUN_ID)

In [0]:
%sql
SELECT *
FROM workspace.flight_tracker.flight_prices_avg;

route_id,airline,airline_code,cabin_class,avg_7day_price,min_7day_price,max_7day_price,stddev_7day_price,latest_price,price_drop_pct,price_trend,data_points_count,calculated_at,valid_from,valid_to,is_current
DEL-BOM,SpiceJet,SG,ECONOMY,0.0,0.0,0.0,null,0.0,0.0,STABLE,1,2026-06-29T11:56:26.546Z,2026-06-29T11:56:26.546Z,2026-06-29T12:10:44.538Z,false
DEL-BOM,Starlight Airline,QP,ECONOMY,0.0,0.0,0.0,0.0,0.0,0.0,STABLE,2,2026-06-29T11:56:26.546Z,2026-06-29T11:56:26.546Z,2026-06-29T12:10:44.538Z,false
DEL-BOM,Air India Express,IX,ECONOMY,0.0,0.0,0.0,null,0.0,0.0,STABLE,1,2026-06-29T11:56:26.546Z,2026-06-29T11:56:26.546Z,2026-06-29T12:10:44.538Z,false
DEL-BOM,Air India,AI,ECONOMY,0.0,0.0,0.0,0.0,0.0,0.0,STABLE,7,2026-06-29T11:56:26.546Z,2026-06-29T11:56:26.546Z,2026-06-29T12:10:44.538Z,false
DEL-BOM,IndiGo,6E,ECONOMY,0.0,0.0,0.0,0.0,0.0,0.0,STABLE,8,2026-06-29T11:56:26.546Z,2026-06-29T11:56:26.546Z,2026-06-29T12:10:44.538Z,false
DEL-BLR,Starlight Airline,QP,ECONOMY,0.0,0.0,0.0,0.0,0.0,0.0,STABLE,3,2026-06-29T11:56:26.546Z,2026-06-29T11:56:26.546Z,2026-06-29T12:10:44.538Z,false
DEL-BLR,Air India Express,IX,ECONOMY,0.0,0.0,0.0,null,0.0,0.0,STABLE,1,2026-06-29T11:56:26.546Z,2026-06-29T11:56:26.546Z,2026-06-29T12:10:44.538Z,false
DEL-BLR,Air India,AI,ECONOMY,0.0,0.0,0.0,0.0,0.0,0.0,STABLE,6,2026-06-29T11:56:26.546Z,2026-06-29T11:56:26.546Z,2026-06-29T12:10:44.538Z,false
DEL-BLR,IndiGo,6E,ECONOMY,0.0,0.0,0.0,0.0,0.0,0.0,STABLE,7,2026-06-29T11:56:26.546Z,2026-06-29T11:56:26.546Z,2026-06-29T12:10:44.538Z,false
BOM-MAA,Starlight Airline,QP,ECONOMY,0.0,0.0,0.0,null,0.0,0.0,STABLE,1,2026-06-29T11:56:26.546Z,2026-06-29T11:56:26.546Z,2026-06-29T12:10:44.538Z,false
